In [3]:
# from pymatgen.ext.matproj import MPRester
from mp_api.client import MPRester
import pandas as pd
from pymatgen.core import Structure, Element, PeriodicSite
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

from tqdm import tqdm
# import matgl
# from matgl.ext.ase import PESCalculator, MolecularDynamics, Relaxer
from collections import defaultdict


/blue/hennig/jasongibson/envs/e3nn/lib/python3.9/site-packages/paramiko/pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
/blue/hennig/jasongibson/envs/e3nn/lib/python3.9/site-packages/paramiko/transport.py:253: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,


In [4]:
elements = [
     'H','Li', 'Be', 'B','C','N','O','F', 'Na', 'Mg', 'Al', 'Si','P','S','Cl', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 
    'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As','Se','Br','Rb', 'Sr', 'Y', 'Zr', 
    'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te','I', 'Cs', 
    'Ba', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'At']

# nin = ['B','Be','C','Ge','Li','Mg','Si','Na','N']

In [5]:
par_el = 'Li'

In [23]:
mpr = MPRester(API_KEY)

In [24]:
mpr.summary.search?

Signature:
mpr.summary.search(
    band_gap: 'tuple[float, float] | None' = None,
    chemsys: 'str | list[str] | None' = None,
    crystal_system: 'CrystalSystem | None' = None,
    density: 'tuple[float, float] | None' = None,
    deprecated: 'bool | None' = None,
    e_electronic: 'tuple[float, float] | None' = None,
    e_ionic: 'tuple[float, float] | None' = None,
    e_total: 'tuple[float, float] | None' = None,
    efermi: 'tuple[float, float] | None' = None,
    elastic_anisotropy: 'tuple[float, float] | None' = None,
    elements: 'list[str] | None' = None,
    energy_above_hull: 'tuple[float, float] | None' = None,
    equilibrium_reaction_energy: 'tuple[float, float] | None' = None,
    exclude_elements: 'list[str] | None' = None,
    formation_energy: 'tuple[float, float] | None' = None,
    formula: 'str | list[str] | None' = None,
    g_reuss: 'tuple[float, float] | None' = None,
    g_voigt: 'tuple[float, float] | None' = None,
    g_vrh: 'tuple[float, float] | None' = N

In [29]:
from pymatgen.io.ase import AseAtomsAdaptor

In [36]:
API_KEY = '1QQt0BLqboJqxKBPocZ53qVRLn1MxBnc'
data = []

with MPRester(API_KEY) as mpr:
    results = mpr.summary.search(
        elements=["Li"],
        energy_above_hull=(-10, 0.1),
        is_metal=True,
        theoretical=False,
        num_elements=(2, 40),
        fields=[
            "material_id", "formula_pretty", "energy_above_hull",
            "formation_energy_per_atom", "band_gap", "density",
            "nsites", "is_magnetic", "structure"
        ],
        deprecated=False,
        chunk_size=100
    )

    for doc in results:
        # Convert pymatgen Structure to ASE Atoms
        ase_atoms = AseAtomsAdaptor.get_atoms(doc.structure)

        data.append({
            "material_id": doc.material_id,
            "formula": doc.formula_pretty,
            "energy_above_hull": doc.energy_above_hull,
            "formation_energy_per_atom": doc.formation_energy_per_atom,
            "band_gap": doc.band_gap,
            "density": doc.density,
            "nsites": doc.nsites,
            "is_magnetic": doc.is_magnetic,
            "ase_atoms": ase_atoms,
            "structure": doc.structure
        })

# Create and save the DataFrame
df = pd.DataFrame(data)

/scratch/local/66163397/ipykernel_3277739/1703154573.py:5: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  results = mpr.summary.search(


Retrieving SummaryDoc documents:   0%|          | 0/464 [00:00<?, ?it/s]

In [37]:
df.to_pickle('test.pkl')

In [38]:
# with MPRester('7Hz5tMiUiiCuu6Jvp5K') as mpr:
#     # d= mpr.query(material_ids=["mp-11557"])
#     # data = mpr.query(criteria={"elements": {"$in": elements,"$nin":nin},"theoretical":False,"nsites":{"$lte":12}}, properties=["original_task_id","e_above_hull", "pretty_formula", 'structure','spacegroup.number','theoretical','nsites','band_gap','nelements'])
#     # data = mpr.query(criteria={"elements": {"$in": elements},"theoretical":False,"nsites":{"$lte":12}}, properties=["original_task_id","e_above_hull", "pretty_formula", 'structure','spacegroup.number','theoretical','nsites','band_gap','nelements'])

In [39]:
# df = pd.DataFrame(data)
# df = df.set_index(keys='original_task_id')
df = df.sort_values('energy_above_hull')

In [40]:
df.head()

,material_id,formula,energy_above_hull,formation_energy_per_atom,band_gap,density,nsites,is_magnetic,ase_atoms,structure
0,mp-569073,LiSn,0.0,-0.325867,0.0,5.092957,6,False,"(Atom('Li', [1.56926692, 3.6093226238413156, 4...","[[1.56926692 3.60932262 4.30354952] Li, [ 1.56..."
273,mp-1078460,Li2CuSn2,0.0,-0.310138,0.0,5.568733,10,False,"(Atom('Li', [0.0, 0.0, 15.977317291287308], ma...","[[ 0. 0. 15.97731729] Li, [0...."
271,mp-644389,Li2H2Pd,0.0,-0.477492,0.0,4.185945,5,False,"(Atom('Li', [0.0, 0.0, 6.604963247987355], mag...","[[0. 0. 6.60496325] Li, [0. ..."
270,mp-7661,Li2RhF6,0.0,-2.415817,0.0,4.045317,18,True,"(Atom('Li', [0.0, 0.0, 5.999554909710947], mag...","[[0. 0. 5.99955491] Li, [2.293..."
269,mp-555112,Li2CrF6,0.0,-2.946233,0.0,3.308625,18,True,"(Atom('Li', [-0.0011880286579049, 0.0416569935...",[[-1.18802866e-03 4.16569935e-02 2.93561748e...


## new criteria

In [41]:
def get_elements(df):
    elements = df.structure.composition.elements
    elements = [ele.symbol for ele in elements]
    return elements
df['elements'] = df.apply(get_elements,axis=1)

In [42]:
lanthanides = [
    "La", "Ce", "Pr", "Nd", "Pm", "Sm", "Eu", 
    "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb", "Lu"
]
nin = ['B','Be','C','Ge','Li','Mg','Si','Na','N']

actinides = [
    "Ac", "Th", "Pa", "U", "Np", "Pu", "Am", 
    "Cm", "Bk", "Cf", "Es", "Fm", "Md", "No", "Lr"
]

In [43]:
# df = df[~df['elements'].apply(lambda x: any(elem in nin for elem in x))]
df = df[~df['elements'].apply(lambda x: any(elem in lanthanides for elem in x))]
df = df[~df['elements'].apply(lambda x: any(elem in actinides for elem in x))]

df.shape

(359, 11)

In [44]:
df

,material_id,formula,energy_above_hull,formation_energy_per_atom,band_gap,density,nsites,is_magnetic,ase_atoms,structure,elements
0,mp-569073,LiSn,0.000000,-0.325867,0.0,5.092957,6,False,"(Atom('Li', [1.56926692, 3.6093226238413156, 4...","[[1.56926692 3.60932262 4.30354952] Li, [ 1.56...","[Li, Sn]"
273,mp-1078460,Li2CuSn2,0.000000,-0.310138,0.0,5.568733,10,False,"(Atom('Li', [0.0, 0.0, 15.977317291287308], ma...","[[ 0. 0. 15.97731729] Li, [0....","[Li, Cu, Sn]"
271,mp-644389,Li2H2Pd,0.000000,-0.477492,0.0,4.185945,5,False,"(Atom('Li', [0.0, 0.0, 6.604963247987355], mag...","[[0. 0. 6.60496325] Li, [0. ...","[Li, H, Pd]"
270,mp-7661,Li2RhF6,0.000000,-2.415817,0.0,4.045317,18,True,"(Atom('Li', [0.0, 0.0, 5.999554909710947], mag...","[[0. 0. 5.99955491] Li, [2.293...","[Li, Rh, F]"
269,mp-555112,Li2CrF6,0.000000,-2.946233,0.0,3.308625,18,True,"(Atom('Li', [-0.0011880286579049, 0.0416569935...",[[-1.18802866e-03 4.16569935e-02 2.93561748e...,"[Li, Cr, F]"
...,...,...,...,...,...,...,...,...,...,...,...
32,mp-1191883,LiTl,0.090757,-0.164843,0.0,7.627105,24,False,"(Atom('Li', [-3.174752500000006, 3.70010579093...","[[-3.1747525 3.70010579 6.09974025] Li, [-3...","[Li, Tl]"
266,mp-555660,Li2V6O13,0.091119,-2.427299,0.0,3.561613,21,True,"(Atom('Li', [0.0005692254239999058, 1.08788709...",[[5.69225424e-04 1.08788710e+00 1.31250321e+00...,"[Li, V, O]"
248,mp-21013,Cs2LiFe(CN)6,0.096898,-0.260311,0.0,2.670388,16,True,"(Atom('Cs', [2.660784, 2.660784, 2.660784], ma...","[[2.660784 2.660784 2.660784] Cs, [7.982352 7....","[Cs, Li, Fe, C, N]"
337,mp-504760,Li10Pb3,0.098659,-0.221232,0.0,4.441533,52,False,"(Atom('Li', [1.1269172805749998, 1.12691728057...","[[1.12691728 1.12691728 1.12691728] Li, [8.983...","[Li, Pb]"


## Old Criteria

In [45]:
# # df_metal = df.loc[df.nsites<=64]
# df_metal = df.loc[df.nsites<=20]
# df_metal = df_metal.loc[df_metal.nelements>1]

In [46]:
# df_metal = df_metal.loc[df_metal.e_above_hull <= 0.2]
# print(df_metal.shape)

In [47]:
elements = [
     'H','Li', 'Be', 'B','C','N','O','F', 'Na', 'Mg', 'Al', 'Si','P','S','Cl', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 
    'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As','Se','Br','Rb', 'Sr', 'Y', 'Zr', 
    'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te','I', 'Cs', 
    'Ba', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'At']
elements = [Element(el) for el in elements]
len(elements)

65

In [48]:
periodic_table = [
    ['H'], 
    ['Li', 'Be', 'B','C','N','O','F'], 
    ['Na', 'Mg', 'Al', 'Si','P','S','Cl'], 
    ['K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As','Se','Br'],
    ['Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te','I'], 
    ['Cs', 'Ba', 'La', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi','Po','At']
]

# Create a dictionary to store the neighbors of each element
element_neighbors = {}

In [49]:
# Iterate through the periodic table to populate the dictionary
for i, row in tqdm(enumerate(periodic_table)):
    for j, element in enumerate(row):
        # Initialize list of neighbors for the current element
        neighbors = []
        neighbors.append(periodic_table[i][j])
        # Add the elements to the left and right
        if j > 0:
            neighbors.append(row[j - 1])  # Add the element to the left
        if j < len(row) - 1:
            neighbors.append(row[j + 1])  # Add the element to the right
        
        # Add the elements above (if applicable)
        if i > 0:
            if len(periodic_table[i - 1])>j:
                neighbors.append(periodic_table[i - 1][j])  # Add the element above
                if j>0:
                    neighbors.append(periodic_table[i - 1][j-1])   
                if j<len(periodic_table[i - 1])-1:
                    neighbors.append(periodic_table[i - 1][j+1])                      
        
        # Add the elements below (if applicable)
        if i < len(periodic_table) - 1:
            neighbors.append(periodic_table[i + 1][j])  # Add the element below
            if j>0:
                neighbors.append(periodic_table[i + 1][j-1])   
            if j<len(periodic_table[i + 1])-1:
                neighbors.append(periodic_table[i + 1][j+1])               
        
        # Store the list of neighbors in the dictionary
        element_neighbors[element] = neighbors


6it [00:00, 31457.28it/s]


In [50]:
element_neighbors['B'] =  ['B', 'Al', 'Mg', 'Si']
element_neighbors['Mg'] =  ['Mg', 'Na', 'Be', 'Li', 'Ca', 'K', 'Sc']
element_neighbors['Al'] =  ['Al', 'Si', 'B','C','Zn','Ga','Ge' ]
element_neighbors['Si'] =  ['Si', 'Al', 'P', 'B', 'C','N', 'Ga', 'Ge', 'As']
element_neighbors['P'] = ['P', 'Si', 'S', 'N', 'C', 'O', 'Ge', 'As', 'Se']
element_neighbors['S'] = ['S', 'P', 'Cl', 'O', 'N', 'F', 'As', 'Se', 'Br']
element_neighbors['Cl'] = ['Cl', 'S', 'F', 'O', 'Se', 'Br']
element_neighbors['Ca'] =  ['Ca', 'K', 'Sc', 'Mg', 'Na', 'Sr', 'Rb', 'Y']
element_neighbors['Ga'] =  ['Ga', 'Zn', 'Ge', 'In', 'Cd', 'Sn','Al','Si']
element_neighbors['Ge'] =  ['Ge', 'Ga', 'As', 'Sn', 'In', 'Sb','Al','Si','P']
element_neighbors['As'] =  ['As', 'Ge','Se', 'Sb', 'Sn', 'Te','Si','P','S']
element_neighbors['Se'] = ['Se', 'As', 'Br', 'Te', 'Sb', 'I', 'P', 'S', 'Cl']

In [51]:
element_neighbors['Sc']= ['Sc', 'Ca', 'Ti', 'Mg', 'Y', 'Sr', 'Zr']

In [52]:
element_neighbors['Re']

['Re', 'W', 'Os', 'Tc', 'Mo', 'Ru']

In [53]:
import numpy as np

In [54]:
from collections import defaultdict
from pymatgen.core import Element, PeriodicSite
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from itertools import product
from tqdm.notebook import tqdm
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.core import Structure

In [55]:
# Helper function to replace elements in a structure
def replace_elements(structure, wyckoff_sites, replacements):
    new_structure = structure.copy()
    for site_idx, site in enumerate(structure):
        if wyckoff_sites[site_idx] in replacements:
            new_element = Element(replacements[wyckoff_sites[site_idx]])
            new_site = PeriodicSite(new_element, site.frac_coords, site.lattice)
            new_structure[site_idx] = new_site
    return new_structure

i = 0
j = 0
new_structures = []
mp_ids = []
formulas = []
parent_formulas = []
sgn = []

# structure = Structure.from_file('POSCAR')
for index, row in df.iterrows():
    structure = row.structure
    if all(elem in elements for elem in structure.species):
    
        formula = structure.composition.reduced_formula
        analyzer = SpacegroupAnalyzer(structure)
        sym_data = analyzer.get_symmetry_dataset()
        wyckoff_sites = sym_data["wyckoffs"]
        wyckoff_species = defaultdict(list)
        
        for site, wyckoff in zip(structure, wyckoff_sites):
            wyckoff_species[wyckoff].append(site.species_string)
    
        sub_sites = [(species_list[0], wyckoff) for wyckoff, species_list in wyckoff_species.items()]
        sub_sites_len = len(sub_sites)
        
        if 1 <= sub_sites_len<=5:
            if sub_sites_len >4:
                print(len(mp_ids))
            wyckoff_positions = [sub_sites[k][1] for k in range(sub_sites_len)]
            elements_list = [element_neighbors[sub_sites[k][0]] for k in range(sub_sites_len)]
            
            for replacement_combination in product(*elements_list):
                replacements = {wyckoff_positions[idx]: replacement_combination[idx] for idx in range(sub_sites_len)}
                new_structure = replace_elements(structure, wyckoff_sites, replacements)
                new_structures.append(new_structure)
                mp_ids.append(1)
                parent_formulas.append(formula)
                atoms = AseAtomsAdaptor.get_atoms(new_structure)
                formulas.append(atoms.get_chemical_formula())
                
                # formulas.append(new_structure.composition.to_pretty_string())
                # sgn.append(row['spacegroup.number'])
                i += 1
    
    
    print(j)
    print(i)

/blue/hennig/jasongibson/envs/e3nn/lib/python3.9/site-packages/spglib/spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['wyckoffs']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instead
  warnings.warn(


0
2025
0
2055
0
2100
0
2820
0
2865
0
6510
0
6510
0
6555
0
6560
0
6565
0
6765
0
6810
0
6840
0
7020
0
7500
0
7509
0
7514
0
7544
0
7704
0
7749
0
7830
0
9180
0
11205
0
12555
0
13905
0
15930
0
17955
17955
0
21555
0
21675
0
21945
0
22215
22215
0
30315
30315
0
38415
0
38445
0
38475
0
39555
0
40635
0
41195
0
41315
0
41720
0
41725
0
41730
0
41739
0
41745
0
41751
0
41976
0
43866
43866
0
50346
50346
0
56826
56826
0
63306
0
64926
0
65196
0
66276
0
67236
0
67281
0
67497
0
67522
0
67672
0
67822
0
67972
0
67997
0
68022
0
68047
0
68077
0
68257
0
68287
0
68317
68317
0
82897
82897
0
89377
0
90277
0
91897
0
92647
92647
0
104797
0
105877
0
106957
0
107002
0
107902
0
108127
0
108352
0
108622
0
109027
0
109432
0
109582
0
109807
0
110032
0
111032
0
111157
0
111157
0
111182
0
111212
0
111217
0
111217
0
111342
0
112782
0
113862
0
113862
0
113862
0
114087
0
114112
114112
0
117237
0
117862
0
118177
0
118447
0
120472
0
121672
0
123022
0
124822
0
126397
0
128422
0
129622
0
131197
0
132247
0
133597
0
135397
0
13674

spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.


0
308942
0
309212
0
309242
0
309452
0
309458
0
309863
0
309868
0
310138
0
310543
0
312118
0
312124
312124
0
321844
0
321849
0
322254
0
324144
0
324159
0
324159
0
326589
0
326994
0
326999
0
327404
0
329429
329429
0
335554
0
335584
0
335614
0
336364
0
336370
0
336615
0
336615
0
336621
0
339456
0
339771
0
340521
0
340551
0
341271
0
341896
0
342301


In [52]:
with open("Fromulas.txt", "w") as file:
    for item in formulas:
        file.write(f"{item}\n")

In [25]:
# Helper function to replace elements in a structure
def replace_elements(structure, wyckoff_sites, replacements):
    new_structure = structure.copy()
    for site_idx, site in enumerate(structure):
        if wyckoff_sites[site_idx] in replacements:
            new_element = Element(replacements[wyckoff_sites[site_idx]])
            new_site = PeriodicSite(new_element, site.frac_coords, site.lattice)
            new_structure[site_idx] = new_site
    return new_structure

i = 0
j = 0
new_structures = []
mp_ids = []
formulas = []
parent_formulas = []
sgn = []

for index, row in tqdm(df.iterrows(),total = df.shape[0]):
    structure = row.structure 
    j += 1    
    if all(elem in elements for elem in structure.species):

        formula = row.pretty_formula
        analyzer = SpacegroupAnalyzer(structure)
        sym_data = analyzer.get_symmetry_dataset()
        wyckoff_sites = sym_data["wyckoffs"]
        wyckoff_species = defaultdict(list)
        
        for site, wyckoff in zip(structure, wyckoff_sites):
            wyckoff_species[wyckoff].append(site.species_string)

        sub_sites = [(species_list[0], wyckoff) for wyckoff, species_list in wyckoff_species.items()]
        sub_sites_len = len(sub_sites)
        
        if 1 <= sub_sites_len<=5:
            if sub_sites_len >4:
                print(len(mp_ids))
            wyckoff_positions = [sub_sites[k][1] for k in range(sub_sites_len)]
            elements_list = [element_neighbors[sub_sites[k][0]] for k in range(sub_sites_len)]
            
            for replacement_combination in product(*elements_list):
                replacements = {wyckoff_positions[idx]: replacement_combination[idx] for idx in range(sub_sites_len)}
                new_structure = replace_elements(structure, wyckoff_sites, replacements)
                new_structures.append(new_structure)
                mp_ids.append(index)
                parent_formulas.append(formula)
                formulas.append(new_structure.composition.reduced_formula)
                sgn.append(row['spacegroup.number'])
                i += 1


    print(j)
    print(i)

  0%|          | 0/225 [00:00<?, ?it/s]

1
486
2
972
3
2916
2916
4
10692
5
12636
6
13932
7
15228
8
15714
9
16038
10
16362
11
16542
12
16812
13
17244
14
17622
15
18054
16
18306
17
18630
18
18846
19
19062
20
19386
21
19872
22
20088
23
20304
24
20592
25
20916
26
22860
27
22914
28
22968
29
23184
30
23220
31
23544
32
23580
33
23904
34
24120
35
25416
36
25740
37
25746
38
25962
39
25998
40
26052
41
27996
42
28320
43
28374
44
28806
45
29130
46
29130
47
29130
48
30426
49
30750
50
32694
51
34206
52
34494
53
36006
54
37950
55
39894
56
41838
57
43782
58
45726
59
45906
60
46230
61
46554
62
46770
63
46986
64
47202
65
47490
66
47814
67
48138
68
48462
69
48678
70
49002
71
49326
72
49812
73
50136
74
50460
75
50784
76
51000
77
51324
78
51648
79
51864
80
52188
81
52512
82
52764
83
52818
84
53142
85
53466
86
53790
87
54078
88
54366
89
54690
90
54906
91
55230
92
55482
93
55806
94
56292
95
56778
96
56783
97
56819
56819
98
64595
99
64919
100
65135
101
65459
102
65495
103
65747
104
65801
105
66017
106
66071
107
66557
108
66845
109
68573
110
70517
11

In [57]:
len(mp_ids)

342301

In [58]:
len(new_structures)

342301

In [60]:
len(new_structures)

342301

In [61]:
len(sgn)

0

In [62]:
df = pd.DataFrame({'structure':new_structures,'mp_id': mp_ids,'parent_formula':parent_formulas,'formula':formulas})

In [68]:
tqdm.pandas()
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.ase import AseAtomsAdaptor as aaa
def get_sgn(df):
    analyzer = SpacegroupAnalyzer(df.structure)
    sym_data = analyzer.get_symmetry_dataset()
    return sym_data['number']
df['sgn'] = df.progress_apply(get_sgn,axis=1)

  0%|          | 0/342301 [00:00<?, ?it/s]

/blue/hennig/jasongibson/envs/e3nn/lib/python3.9/site-packages/spglib/spglib.py:115: DeprecationWarning: dict interface (SpglibDataset['number']) is deprecated.Use attribute interface ({self.__class__.__name__}.{key}) instead
  warnings.warn(
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.
spglib: ssm_get_exact_positions failed.
spglib: get_bravais_exact_positions_and_lattice failed.


In [69]:
df_cleaned = df.drop_duplicates(subset=['sgn', 'formula'], keep='first')
# df_og = pd.read_pickle(f'pickle_files/{par_el}_sub.pkl')

In [70]:
df_cleaned.reset_index(inplace=True)

In [31]:
par_el = 'temp'

In [74]:
df = df_cleaned.loc[df_cleaned.formula.str.contains('Li')]

In [75]:
df.to_pickle(f'Li.pkl')

In [76]:
df

,index,structure,mp_id,parent_formula,formula,sgn
0,0,"[[1.56926692 3.60932262 4.30354952] Li, [ 1.56...",1,LiSn,Li3Sn3,10
1,1,"[[1.56926692 3.60932262 4.30354952] Li, [ 1.56...",1,LiSn,InLi3Sn2,10
2,2,"[[1.56926692 3.60932262 4.30354952] Li, [ 1.56...",1,LiSn,Li3SbSn2,10
3,3,"[[1.56926692 3.60932262 4.30354952] Li, [ 1.56...",1,LiSn,GeLi3Sn2,10
4,4,"[[1.56926692 3.60932262 4.30354952] Li, [ 1.56...",1,LiSn,GaLi3Sn2,10
...,...,...,...,...,...,...
170261,341972,"[[0. 0. 3.2541605] Li, [0. 0. 0....",1,LiSnS2,BiLiN2,164
170262,341973,"[[0. 0. 3.2541605] Li, [0. 0. 0....",1,LiSnS2,BiF2Li,164
170263,341974,"[[0. 0. 3.2541605] Li, [0. 0. 0....",1,LiSnS2,As2BiLi,164
170264,341975,"[[0. 0. 3.2541605] Li, [0. 0. 0....",1,LiSnS2,BiLiSe2,164


In [33]:
df_cleaned.shape[0]/1000

56.368

In [34]:
from tqdm.notebook import tqdm

In [35]:
root = f'/blue/hennig/jasongibson/elemental_sub/materials/{par_el}/unrelaxed/'
for index, row in tqdm(df_cleaned.iterrows(),total=len(df_cleaned)):
    row.structure.to(root + f'POSCAR_{index}')


  0%|          | 0/56368 [00:00<?, ?it/s]

In [233]:
for struct in tqdm(new_structures):
    relax_results = relaxer.relax(struct, fmax=0.001)
    # extract results
    final_structure = relax_results["final_structure"]
    final_energy = relax_results["trajectory"].energies[-1]
    # print out the final relaxed structure and energy

    print(f"The final energy is {float(final_energy):.3f} eV.")

    # For multi-fidelity models, we need to define graph label ("0": PBE, "1": GLLB-SC, "2": HSE, "3": SCAN)
    for i, method in ((0, "PBE"), (1, "GLLB-SC"), (2, "HSE"), (3, "SCAN")):
        graph_attrs = torch.tensor([i])
        bandgap = model_bg.predict_structure(structure=final_structure, state_attr=graph_attrs)
        print(f"The predicted {method} band gap for {final_structure.composition.reduced_formula} is {float(bandgap):.3f} eV.")

    eform = model_form.predict_structure(final_structure)
    print(f"The predicted formation energy for {final_structure.composition.reduced_formula} is {float(eform):.3f} eV/atom.")    


  0%|                                                                                                                                                     | 1/9610 [00:02<6:48:52,  2.55s/it]

The final energy is -33.480 eV.
The predicted PBE band gap for BeNbRu2 is -0.008 eV.
The predicted GLLB-SC band gap for BeNbRu2 is 0.629 eV.
The predicted HSE band gap for BeNbRu2 is -0.008 eV.
The predicted SCAN band gap for BeNbRu2 is 0.146 eV.
The predicted formation energy for BeNbRu2 is -0.281 eV/atom.



  0%|                                                                                                                                                    | 2/9610 [00:07<10:01:41,  3.76s/it]

The final energy is -35.227 eV.
The predicted PBE band gap for BeNbTc2 is -0.008 eV.
The predicted GLLB-SC band gap for BeNbTc2 is 0.980 eV.
The predicted HSE band gap for BeNbTc2 is -0.008 eV.
The predicted SCAN band gap for BeNbTc2 is 0.497 eV.
The predicted formation energy for BeNbTc2 is -0.167 eV/atom.



  0%|                                                                                                                                                    | 3/9610 [00:15<15:06:27,  5.66s/it]

...

The final energy is -35.070 eV.
The predicted PBE band gap for Mn2BeW is -0.008 eV.
The predicted GLLB-SC band gap for Mn2BeW is 0.547 eV.
The predicted HSE band gap for Mn2BeW is -0.004 eV.
The predicted SCAN band gap for Mn2BeW is 0.215 eV.
The predicted formation energy for Mn2BeW is -0.052 eV/atom.



  1%|█▏                                                                                                                                                 | 78/9610 [13:33<28:40:14, 10.83s/it]

The final energy is -31.392 eV.
The predicted PBE band gap for BeCo2W is -0.006 eV.
The predicted GLLB-SC band gap for BeCo2W is 0.792 eV.
The predicted HSE band gap for BeCo2W is -0.009 eV.
The predicted SCAN band gap for BeCo2W is 0.058 eV.
The predicted formation energy for BeCo2W is -0.149 eV/atom.


  1%|█▏                                                                                                                                                 | 78/9610 [13:39<27:48:17, 10.50s/it]


KeyboardInterrupt: 

In [209]:
for s in new_structures:
    print(s.composition.reduced_formula)

BeNbRu2
BeNbTc2
BeNbRh2
BeNbFe2
Mn2BeNb
BeNbCo2
BeNbOs2
BeNbRe2
BeNbIr2
ZrBeRu2
ZrBeTc2
ZrBeRh2
ZrBeFe2
ZrMn2Be
ZrBeCo2
ZrBeOs2
ZrBeRe2
ZrBeIr2
BeMoRu2
BeTc2Mo
BeMoRh2
BeFe2Mo
Mn2BeMo
BeCo2Mo
BeMoOs2
BeRe2Mo
BeMoIr2
BeVRu2
BeVTc2
BeVRh2
BeVFe2
Mn2BeV
BeVCo2
BeVOs2
BeVRe2
BeVIr2
TiBeRu2
TiBeTc2
TiBeRh2
TiBeFe2
TiMn2Be
TiBeCo2
TiBeOs2
TiBeRe2
TiBeIr2
BeCrRu2
BeCrTc2
BeCrRh2
BeCrFe2
Mn2BeCr
BeCrCo2
BeCrOs2
BeCrRe2
BeCrIr2
TaBeRu2
TaBeTc2
TaBeRh2
TaBeFe2
TaMn2Be
TaBeCo2
TaBeOs2
TaBeRe2
TaBeIr2
HfBeRu2
HfBeTc2
HfBeRh2
HfBeFe2
HfMn2Be
HfBeCo2
HfBeOs2
HfBeRe2
HfBeIr2
BeRu2W
BeTc2W
BeRh2W
BeFe2W
Mn2BeW
BeCo2W
BeOs2W
BeRe2W
BeIr2W
BeSiRu2
BeSiTc2
BeSiRh2
BeFe2Si
Mn2BeSi
BeCo2Si
BeSiOs2
BeRe2Si
BeSiIr2
BeAlRu2
BeAlTc2
BeAlRh2
BeAlFe2
Mn2BeAl
BeAlCo2
BeAlOs2
BeAlRe2
BeAlIr2
TiBeRu2
TiBeTc2
TiBeRh2
TiBeFe2
TiMn2Be
TiBeCo2
TiBeOs2
TiBeRe2
TiBeIr2
ScBeRu2
ScBeTc2
ScBeRh2
ScBeFe2
ScMn2Be
ScBeCo2
ScBeOs2
ScBeRe2
ScBeIr2
BeVRu2
BeVTc2
BeVRh2
BeVFe2
Mn2BeV
BeVCo2
BeVOs2
BeVRe2
BeVIr2
Be12V
TiBe12
Be12Cr

NaBeAs
NaBeSn
NaBeIn
NaBeSb
BeAlGe
BeAlGa
BeAlAs
BeAlSn
BeAlIn
BeAlSb
SrBeGe
SrBeGa
SrBeAs
SrBeSn
SrBeIn
SrBeSb
RbBeGe
RbBeGa
RbBeAs
RbBeSn
RbBeIn
RbBeSb
YBeGe
YBeGa
YBeAs
YBeSn
YBeIn
YBeSb
MgBe4Cu
MgBe4Ni
MgBe4Zn
MgBe4Ag
MgBe4Pd
MgBe4Cd
NaBe4Cu
NaBe4Ni
NaBe4Zn
NaBe4Ag
NaBe4Pd
NaBe4Cd
Be4AlCu
Be4AlNi
Be4AlZn
Be4AlAg
Be4AlPd
Be4AlCd
Be5Cu
Be5Ni
Be5Zn
Be5Ag
Be5Pd
Be5Cd
LiBe4Cu
LiBe4Ni
LiBe4Zn
LiBe4Ag
LiBe4Pd
LiBe4Cd
Be4CuB
Be4NiB
Be4ZnB
Be4AgB
Be4BPd
Be4CdB
CaBe4Cu
CaBe4Ni
CaBe4Zn
CaBe4Ag
CaBe4Pd
CaBe4Cd
KBe4Cu
KBe4Ni
KBe4Zn
KBe4Ag
KBe4Pd
KBe4Cd
ScBe4Cu
ScBe4Ni
ScBe4Zn
ScBe4Ag
ScBe4Pd
ScBe4Cd
Be2B
Be
Be2Al
MgBe2
Be2Si
SrBeSi
SrBeAl
SrTiBe
SrScBe
SrBeV
RbBeSi
RbBeAl
RbTiBe
RbScBe
RbBeV
YBeSi
YBeAl
YTiBe
YScBe
YBeV
CaBeSi
CaBeAl
CaTiBe
CaScBe
CaBeV
KBeSi
KBeAl
KTiBe
KScBe
KBeV
ScBeSi
ScBeAl
ScTiBe
Sc2Be
ScBeV
BaBeSi
BaBeAl
BaTiBe
BaScBe
BaBeV
CsBeSi
CsBeAl
CsTiBe
CsScBe
CsBeV
LaBeSi
LaBeAl
LaTiBe
LaScBe
LaBeV
Be3Cu
Be3Ni
Be3Zn
Be3Ag
Be3Pd
Be3Cd
SrBeGe
SrBeGa
SrBeAs
SrBeSn
SrBeIn
SrBeSb
RbB

MnBe2Rh
MnBe2Fe
MnBe
MnBe2Co
MnBe2Os
MnBe2Re
MnBe2Ir
Be2CrRu
Be2CrTc
Be2CrRh
Be2CrFe
MnBe2Cr
Be2CrCo
Be2CrOs
Be2CrRe
Be2CrIr
Be2FeRu
Be2FeTc
Be2FeRh
BeFe
MnBe2Fe
Be2FeCo
Be2FeOs
Be2FeRe
Be2FeIr
Be2TcRu
BeTc
Be2TcRh
Be2FeTc
MnBe2Tc
Be2CoTc
Be2TcOs
Be2ReTc
Be2TcIr
Be2MoRu
Be2TcMo
Be2MoRh
Be2FeMo
MnBe2Mo
Be2CoMo
Be2MoOs
Be2ReMo
Be2MoIr
BeRu
Be2TcRu
Be2RuRh
Be2FeRu
MnBe2Ru
Be2CoRu
Be2OsRu
Be2ReRu
Be2IrRu
BeFeCo2
BeFe3
BeFeNi2
BeFeRh2
BeFeRu2
BeFePd2
MnBeCo2
MnBeFe2
MnBeNi2
MnBeRh2
MnBeRu2
MnBePd2
BeCo3
BeFe2Co
BeCoNi2
BeCoRh2
BeCoRu2
BeCoPd2
BeCo2Ru
BeFe2Ru
BeNi2Ru
BeRuRh2
BeRu3
BePd2Ru
BeCo2Tc
BeFe2Tc
BeTcNi2
BeTcRh2
BeTcRu2
BeTcPd2
BeCo2Rh
BeFe2Rh
BeNi2Rh
BeRh3
BeRu2Rh
BePd2Rh
TaBeRh2
TaBeRu2
TaBePd2
TaBeCo2
TaBeFe2
TaBeNi2
TaBeIr2
TaBeOs2
TaBePt2
HfBeRh2
HfBeRu2
HfBePd2
HfBeCo2
HfBeFe2
HfBeNi2
HfBeIr2
HfBeOs2
HfBePt2
BeRh2W
BeRu2W
BePd2W
BeCo2W
BeFe2W
BeNi2W
BeIr2W
BeOs2W
BePt2W
BeNbRh2
BeNbRu2
BeNbPd2
BeNbCo2
BeNbFe2
BeNbNi2
BeNbIr2
BeNbOs2
BeNbPt2
ZrBeRh2
ZrBeRu2
ZrBePd2
ZrBeCo2
ZrBeF

In [105]:
from pymatgen.analysis.structure_prediction.substitution_probability import SubstitutionPredictor
from pymatgen.core import Composition

In [106]:
sub = SubstitutionPredictor()

In [108]:
c = Composition('Si3N4')

In [114]:
from pymatgen.analysis.structure_prediction import OxidationStateGuess

ImportError: cannot import name 'OxidationStateGuess' from 'pymatgen.analysis.structure_prediction' (/home/jasongibson/.conda/envs/e3nn/lib/python3.9/site-packages/pymatgen/analysis/structure_prediction/__init__.py)

In [107]:
sub.composition_prediction(Composition('Si3N4'))

ValueError: the species Si is not allowed for the probability model you are using

In [100]:
i

79448526

In [95]:
i

96762060

In [92]:
i

200402220

In [89]:
i

209163500

In [59]:
from collections import defaultdict

wyckoff_species = defaultdict(list)
for site, wyckoff in zip(structure, wyckoff_sites):
    wyckoff_species[wyckoff].append(site.species_string)
    
sub_sites = []
for wyckoff, species_list in wyckoff_species.items():
    if 'Be' not in species_list:
        sub_sites.append(wyckoff)
        print(f"Wyckoff site {wyckoff}: {species_list}")


Wyckoff site a: ['Nd', 'Nd']


In [75]:
elements = [
    'H', 'Li', 'Be', 'B', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 
    'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As','Rb', 'Sr', 'Y', 'Zr', 
    'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'Cs', 
    'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 
    'Yb', 'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'At']
len(elements)

70

In [61]:
from pymatgen.core.sites import PeriodicSite
from pymatgen.core import Element

# Create a copy of the structure to modify
new_structure = structure.copy()

# Iterate over the sites and substitute based on Wyckoff sites
for i, site in enumerate(structure):
    if wyckoff_sites[i] == "g":  # Replace "a" with the specific Wyckoff position
        new_element = Element("Cl")  # Replace "Ge" with the desired element
        new_site = PeriodicSite(new_element, site.frac_coords, site.lattice)
        new_structure[i] = new_site

# Print the modified structure
print(new_structure)

# Save the modified structure to a CIF file


Full Formula (Nd2 Be26)
Reduced Formula: NdBe13
abc   :   7.284912   7.284912   7.284912
angles:  60.000000  60.000000  60.000000
pbc   :       True       True       True
Sites (28)
  #  SP           a         b         c    magmom
---  ----  --------  --------  --------  --------
  0  Nd    0.75      0.75      0.75            -0
  1  Nd    0.25      0.25      0.25             0
  2  Be    0.788495  0.563061  0.436939        -0
  3  Be    0.563061  0.788495  0.211505         0
  4  Be    0.211505  0.436939  0.563061         0
  5  Be    0.436939  0.211505  0.788495         0
  6  Be    0.436939  0.788495  0.563061        -0
  7  Be    0.563061  0.436939  0.788495        -0
  8  Be    0.788495  0.436939  0.211505        -0
  9  Be    0.788495  0.211505  0.563061        -0
 10  Be    0.211505  0.563061  0.788495         0
 11  Be    0.436939  0.563061  0.211505        -0
 12  Be    0.563061  0.211505  0.436939         0
 13  Be    0.211505  0.788495  0.436939         0
 14  Be    0.93693

In [217]:
from tqdm.notebook import tqdm
import matgl
from matgl.ext.ase import PESCalculator, MolecularDynamics, Relaxer

pot = matgl.load_model("M3GNet-MP-2021.2.8-PES")
model_form = matgl.load_model("MEGNet-MP-2018.6.1-Eform")
model_bg = matgl.load_model("MEGNet-MP-2019.4.1-BandGap-mfi")

relaxer = Relaxer(potential=pot)
relax_results = relaxer.relax(new_structures[1], fmax=0.001)
# extract results
final_structure = relax_results["final_structure"]
final_energy = relax_results["trajectory"].energies[-1]
# print out the final relaxed structure and energy
print(final_structure)
print(f"The final energy is {float(final_energy):.3f} eV.")

# For multi-fidelity models, we need to define graph label ("0": PBE, "1": GLLB-SC, "2": HSE, "3": SCAN)
for i, method in ((0, "PBE"), (1, "GLLB-SC"), (2, "HSE"), (3, "SCAN")):
    graph_attrs = torch.tensor([i])
    bandgap = model_bg.predict_structure(structure=final_structure, state_attr=graph_attrs)
    print(f"The predicted {method} band gap for {final_structure.composition.reduced_formula} is {float(bandgap):.3f} eV.")

eform = model_form.predict_structure(final_structure)
print(f"The predicted formation energy for {final_structure.composition.reduced_formula} is {float(eform):.3f} eV/atom.")    

In [215]:
relaxer = Relaxer(potential=pot)
relax_results = relaxer.relax(new_structures[1], fmax=0.001)
# extract results
final_structure = relax_results["final_structure"]
final_energy = relax_results["trajectory"].energies[-1]
# print out the final relaxed structure and energy
print(final_structure)
print(f"The final energy is {float(final_energy):.3f} eV.")

# For multi-fidelity models, we need to define graph label ("0": PBE, "1": GLLB-SC, "2": HSE, "3": SCAN)
for i, method in ((0, "PBE"), (1, "GLLB-SC"), (2, "HSE"), (3, "SCAN")):
    graph_attrs = torch.tensor([i])
    bandgap = model_bg.predict_structure(structure=final_structure, state_attr=graph_attrs)
    print(f"The predicted {method} band gap for {final_structure.composition.reduced_formula} is {float(bandgap):.3f} eV.")

eform = model_form.predict_structure(final_structure)
print(f"The predicted formation energy for {final_structure.composition.reduced_formula} is {float(eform):.3f} eV/atom.")    

Full Formula (Be1 Nb1 Tc2)
Reduced Formula: BeNbTc2
abc   :   4.317669   4.317670   4.317670
angles:  60.000003  60.000006  59.999996
pbc   :       True       True       True
Sites (4)
  #  SP        a     b      c    magmom
---  ----  -----  ----  -----  --------
  0  Be     0.5   0.5    0.5         -0
  1  Nb    -0     0     -0          nan
  2  Tc     0.25  0.25   0.25       nan
  3  Tc     0.75  0.75   0.75       nan
The final energy is -35.227 eV.


In [219]:
# For multi-fidelity models, we need to define graph label ("0": PBE, "1": GLLB-SC, "2": HSE, "3": SCAN)
for i, method in ((0, "PBE"), (1, "GLLB-SC"), (2, "HSE"), (3, "SCAN")):
    graph_attrs = torch.tensor([i])
    bandgap = model_bg.predict_structure(structure=final_structure, state_attr=graph_attrs)
    print(f"The predicted {method} band gap for {final_structure.composition.reduced_formula} is {float(bandgap):.3f} eV.")

eform = model_form.predict_structure(final_structure)
print(f"The predicted formation energy for {final_structure.composition.reduced_formula} is {float(eform):.3f} eV/atom.")    

The predicted PBE band gap for BeNbTc2 is -0.008 eV.
The predicted GLLB-SC band gap for BeNbTc2 is 0.980 eV.
The predicted HSE band gap for BeNbTc2 is -0.008 eV.
The predicted SCAN band gap for BeNbTc2 is 0.497 eV.
The predicted formation energy for BeNbTc2 is -0.167 eV/atom.


/home/jasongibson/.conda/envs/e3nn/lib/python3.9/site-packages/dgl/readout.py:443: DGLWarning:

For a single graph, use a tensor of shape (1, *) for graph_feat. The support of shape (*) will be deprecated.



In [63]:
len(final_structure)

144

In [64]:
new_structure =new_structure.make_supercell([2,2,2])

In [211]:
new_structures[1]

Structure Summary
Lattice
    abc : 4.280593936492692 4.280593936492692 4.280593936492692
 angles : 60.00000000000001 60.00000000000001 60.00000000000001
 volume : 55.46220069957552
      A : 0.0 3.026837 3.026837
      B : 3.026837 0.0 3.026837
      C : 3.026837 3.026837 0.0
    pbc : True True True
PeriodicSite: Be (3.027, 3.027, 3.027) [0.5, 0.5, 0.5]
PeriodicSite: Nb (0.0, 0.0, 0.0) [0.0, 0.0, 0.0]
PeriodicSite: Tc (1.513, 1.513, 1.513) [0.25, 0.25, 0.25]
PeriodicSite: Tc (4.54, 4.54, 4.54) [0.75, 0.75, 0.75]

In [210]:
relaxer = Relaxer(potential=pot)
relax_results = relaxer.relax(new_structures[1], fmax=0.001)
# extract results
final_structure = relax_results["final_structure"]
final_energy = relax_results["trajectory"].energies[-1]
# print out the final relaxed structure and energy

print(final_structure)
print(f"The final energy is {float(final_energy):.3f} eV.")


/home/jasongibson/.conda/envs/e3nn/lib/python3.9/site-packages/matgl/layers/_basis.py:121: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).



Full Formula (Be1 Nb1 Tc2)
Reduced Formula: BeNbTc2
abc   :   4.317669   4.317670   4.317670
angles:  60.000003  60.000006  59.999996
pbc   :       True       True       True
Sites (4)
  #  SP        a     b      c    magmom
---  ----  -----  ----  -----  --------
  0  Be     0.5   0.5    0.5         -0
  1  Nb    -0     0     -0          nan
  2  Tc     0.25  0.25   0.25       nan
  3  Tc     0.75  0.75   0.75       nan
The final energy is -35.227 eV.


In [66]:
analyzer = SpacegroupAnalyzer(final_structure)
sym_data = analyzer.get_symmetry_dataset()

In [67]:
final_structure = relax_results['final_structure']
final_energy_per_atom = float(relax_results['trajectory'].energies[-1] / len(final_structure))

print(f"Relaxed lattice parameter is {final_structure.lattice.abc[0]:.3f} Å")
print(f"Final energy is {final_energy_per_atom:.3f} eV/atom")

Relaxed lattice parameter is 14.585 Å
Final energy is -3.970 eV/atom


In [68]:
from pymatgen.analysis.phase_diagram import *
from pymatgen.ext.matproj import MPRester

In [70]:
entries = mpr.get_entries_in_chemsys(['Be', 'Nd'])
    
phasediagram = PhaseDiagram(entries)

plotter = PDPlotter(phasediagram)

p = plotter.get_plot(label_unstable=False)
p.show()

In [71]:
pde = PDEntry(final_structure.composition,final_energy)
phasediagram.get_form_energy_per_atom(pde)

-0.15670271390764562

In [72]:
phasediagram.get_e_above_hull(pde)

0.005415243235212142

In [73]:
len(final_structure)

224

In [55]:
import torch

In [74]:
model = matgl.load_model("MEGNet-MP-2019.4.1-BandGap-mfi")

# For multi-fidelity models, we need to define graph label ("0": PBE, "1": GLLB-SC, "2": HSE, "3": SCAN)
for i, method in ((0, "PBE"), (1, "GLLB-SC"), (2, "HSE"), (3, "SCAN")):
    graph_attrs = torch.tensor([i])
    bandgap = model.predict_structure(structure=final_structure, state_attr=graph_attrs)
    print(f"The predicted {method} band gap for CsCl is {float(bandgap):.3f} eV.")


/home/jasongibson/.conda/envs/e3nn/lib/python3.9/site-packages/matgl/utils/io.py:126: UserWarning:

Incompatible model version detected! The code will continue to load the model but it is recommended that you provide a path to an updated model, increment your @model_version in model.json if you are confident that the changes are not problematic, or clear your ~/.matgl cache using `python -c "import matgl; matgl.clear_cache()"`



The predicted PBE band gap for CsCl is -0.010 eV.
The predicted GLLB-SC band gap for CsCl is 0.114 eV.
The predicted HSE band gap for CsCl is -0.008 eV.
The predicted SCAN band gap for CsCl is 0.019 eV.
